In [5]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

clinical = pd.read_csv("/Users/yasminislam/Downloads/MedExtract-AI/notebooks/archive-5/clinical_notes.csv")
diaries = pd.read_csv("/Users/yasminislam/Downloads/MedExtract-AI/notebooks/archive-5/patient_diaries.csv")

print("Clinical notes:", clinical.shape)
print("Patient diaries:", diaries.shape)
print()
print("Clinical label distribution:")
print(clinical['label'].value_counts())
print()
print("Diaries label distribution:")
print(diaries['label'].value_counts())

Clinical notes: (1000, 7)
Patient diaries: (1000, 7)

Clinical label distribution:
label
depression       605
no depression    395
Name: count, dtype: int64

Diaries label distribution:
label
no depression    514
depression       486
Name: count, dtype: int64


In [6]:
print("=== CLINICAL NOTES sample ===")
for i, row in clinical.sample(5, random_state=42).iterrows():
    print(f"[{row['label']}] {row['text']}")
    print("---")

print("\n=== PATIENT DIARIES sample ===")
for i, row in diaries.sample(5, random_state=42).iterrows():
    print(f"[{row['label']}] {row['text']}")
    print("---")

=== CLINICAL NOTES sample ===
[depression] Patient shows signs of depression, feeling anxious and fatigued.
---
[depression] Patient demonstrated symptoms of depression; further assessment needed.
---
[no depression] Patient feels low; a referral to a psychiatrist may be necessary.
---
[no depression] Patient feels better; a referral to a psychiatrist may be necessary.
---
[depression] Patient indicates a lack of interest in activities, feeling isolated.
---

=== PATIENT DIARIES sample ===
[depression] I felt confused today. I found it hard to focus today; it was a heavy day.
---
[depression] I felt empty today. I found it hard to focus today; it was a heavy day.
---
[depression] I felt confused today. I felt a glimmer of hope today, looking forward to tomorrow.
---
[no depression] I felt down today. I felt a glimmer of hope today, looking forward to tomorrow.
---
[no depression] I felt better today. I found it hard to focus today; it was a heavy day.
---


In [7]:
print("Unique texts in clinical:", clinical['text'].nunique(), "/", len(clinical))
print("Unique texts in diaries:", diaries['text'].nunique(), "/", len(diaries))

print("\n--- Clinical notes with 'no depression' label but concerning language ---")
suspicious = clinical[
    (clinical['label'] == 'no depression') & 
    (clinical['text'].str.contains('low|referral|psychiatrist|isolated|fatigued', case=False))
]
print(f"Found {len(suspicious)} suspicious rows")
for i, row in suspicious.head(5).iterrows():
    print(f"- {row['text']}")

print("\n--- Diary entries with 'no depression' label but concerning language ---")
suspicious2 = diaries[
    (diaries['label'] == 'no depression') & 
    (diaries['text'].str.contains('down|confused|empty|heavy|hard to focus', case=False))
]
print(f"Found {len(suspicious2)} suspicious rows")
for i, row in suspicious2.head(5).iterrows():
    print(f"- {row['text']}")

Unique texts in clinical: 57 / 1000
Unique texts in diaries: 64 / 1000

--- Clinical notes with 'no depression' label but concerning language ---
Found 102 suspicious rows
- Patient expressed feelings of being low and lacks motivation.
- Patient feels unstable; a referral to a psychiatrist may be necessary.
- Patient feels better; a referral to a psychiatrist may be necessary.
- Patient feels overwhelmed; a referral to a psychiatrist may be necessary.
- Patient has been experiencing low thoughts and needs support.

--- Diary entries with 'no depression' label but concerning language ---
Found 183 suspicious rows
- I felt confused today. I felt a glimmer of hope today, looking forward to tomorrow.
- I felt down today. I felt a glimmer of hope today, looking forward to tomorrow.
- I felt down today. I took a walk in the evening, which slightly lifted my spirits.
- I felt down today. I shared a laugh with a friend, which helped me momentarily.
- I felt down today. I felt a bit off but man

In [9]:
clinical_unique = clinical.drop_duplicates(subset='text')[['text', 'label']].reset_index(drop=True)
diaries_unique = diaries.drop_duplicates(subset='text')[['text', 'label']].reset_index(drop=True)

clinical_unique.to_csv("clinical_unique_for_labeling.csv", index=False)
diaries_unique.to_csv("diaries_unique_for_labeling.csv", index=False)

print(clinical_unique.shape, diaries_unique.shape)

(57, 2) (64, 2)


In [10]:
for i, row in clinical_unique.iterrows():
    print(f"{i+1}. [csv label: {row['label']}] {row['text']}")

1. [csv label: depression] Patient demonstrated symptoms of depression; further assessment needed.
2. [csv label: depression] Patient shows signs of depression, feeling low and fatigued.
3. [csv label: depression] Patient has fluctuating moods, feeling sad.
4. [csv label: no depression] Patient indicates a lack of interest in activities, feeling hopeful.
5. [csv label: depression] Patient reports feeling low and unable to concentrate.
6. [csv label: no depression] Patient expressed feelings of being overwhelmed and lacks motivation.
7. [csv label: no depression] Patient indicates a lack of interest in activities, feeling overwhelmed.
8. [csv label: no depression] Patient expressed feelings of being low and lacks motivation.
9. [csv label: no depression] Patient has been experiencing overwhelmed thoughts and needs support.
10. [csv label: no depression] Patient feels unstable; a referral to a psychiatrist may be necessary.
11. [csv label: no depression] Patient feels better; a referral 

In [11]:
for i, row in diaries_unique.iterrows():
    print(f"{i+1}. [csv label: {row['label']}] {row['text']}")

1. [csv label: no depression] I felt happy today. I shared a laugh with a friend, which helped me momentarily.
2. [csv label: depression] I felt fine today. I wrote in my journal and expressed my thoughts.
3. [csv label: depression] I felt confused today. I felt overwhelmed with sadness and isolation.
4. [csv label: no depression] I felt confused today. I felt a glimmer of hope today, looking forward to tomorrow.
5. [csv label: no depression] I felt down today. I felt a glimmer of hope today, looking forward to tomorrow.
6. [csv label: no depression] I felt down today. I took a walk in the evening, which slightly lifted my spirits.
7. [csv label: no depression] I felt down today. I shared a laugh with a friend, which helped me momentarily.
8. [csv label: depression] I felt happy today. I felt overwhelmed with sadness and isolation.
9. [csv label: no depression] I felt happy today. I felt a bit off but managed to accomplish some tasks.
10. [csv label: depression] I felt confused today. 

In [12]:
clinical_gold_labels = [
    "depression","depression","depression","ambiguous","depression",
    "depression","depression","depression","depression","ambiguous",
    "ambiguous","not depression","depression","ambiguous","depression",
    "depression","ambiguous","ambiguous","depression","depression",
    "ambiguous","depression","depression","ambiguous","depression",
    "ambiguous","not depression","depression","ambiguous","depression",
    "depression","depression","ambiguous","depression","depression",
    "ambiguous","ambiguous","depression","depression","depression",
    "depression","ambiguous","depression","depression","ambiguous",
    "depression","depression","depression","depression","ambiguous",
    "depression","ambiguous","depression","depression","depression",
    "depression","ambiguous"
]

assert len(clinical_gold_labels) == len(clinical_unique), f"Mismatch: {len(clinical_gold_labels)} vs {len(clinical_unique)}"
clinical_unique["gold_label"] = clinical_gold_labels
clinical_unique[["gold_label", "label", "text"]].head(10)

,gold_label,label,text
0,depression,depression,Patient demonstrated symptoms of depression; further assessment needed.
1,depression,depression,"Patient shows signs of depression, feeling low and fatigued."
2,depression,depression,"Patient has fluctuating moods, feeling sad."
3,ambiguous,no depression,"Patient indicates a lack of interest in activities, feeling hopeful."
4,depression,depression,Patient reports feeling low and unable to concentrate.
5,depression,no depression,Patient expressed feelings of being overwhelmed and lacks motivation.
6,depression,no depression,"Patient indicates a lack of interest in activities, feeling overwhelmed."
7,depression,no depression,Patient expressed feelings of being low and lacks motivation.
8,depression,no depression,Patient has been experiencing overwhelmed thoughts and needs support.
9,ambiguous,no depression,Patient feels unstable; a referral to a psychiatrist may be necessary.


In [13]:
diary_gold_labels = [
    "not depression","ambiguous","depression","ambiguous","ambiguous",
    "ambiguous","ambiguous","ambiguous","not depression","depression",
    "ambiguous","depression","not depression","not depression","depression",
    "ambiguous","ambiguous","ambiguous","depression","depression",
    "ambiguous","ambiguous","not depression","ambiguous","depression",
    "depression","depression","ambiguous","not depression","ambiguous",
    "ambiguous","not depression","not depression","ambiguous","ambiguous",
    "ambiguous","depression","ambiguous","ambiguous","ambiguous",
    "ambiguous","depression","ambiguous","ambiguous","not depression",
    "ambiguous","ambiguous","not depression","ambiguous","not depression",
    "ambiguous","not depression","ambiguous","depression","depression",
    "ambiguous","not depression","ambiguous","not depression","ambiguous",
    "ambiguous","ambiguous","not depression","not depression"
]

assert len(diary_gold_labels) == len(diaries_unique), f"Mismatch: {len(diary_gold_labels)} vs {len(diaries_unique)}"
diaries_unique["gold_label"] = diary_gold_labels
diaries_unique[["gold_label", "label", "text"]].head(10)

,gold_label,label,text
0,not depression,no depression,"I felt happy today. I shared a laugh with a friend, which helped me momentarily."
1,ambiguous,depression,I felt fine today. I wrote in my journal and expressed my thoughts.
2,depression,depression,I felt confused today. I felt overwhelmed with sadness and isolation.
3,ambiguous,no depression,"I felt confused today. I felt a glimmer of hope today, looking forward to tomorrow."
4,ambiguous,no depression,"I felt down today. I felt a glimmer of hope today, looking forward to tomorrow."
5,ambiguous,no depression,"I felt down today. I took a walk in the evening, which slightly lifted my spirits."
6,ambiguous,no depression,"I felt down today. I shared a laugh with a friend, which helped me momentarily."
7,ambiguous,depression,I felt happy today. I felt overwhelmed with sadness and isolation.
8,not depression,no depression,I felt happy today. I felt a bit off but managed to accomplish some tasks.
9,depression,depression,I felt confused today. I struggled to get out of bed and had no motivation.


In [15]:
clinical_unique.to_csv("clinical_gold_labels.csv", index=False)
diaries_unique.to_csv("diaries_gold_labels.csv", index=False)

def agreement_rate(df):
    non_ambiguous = df[df["gold_label"] != "ambiguous"]
    matches = (non_ambiguous["gold_label"] == non_ambiguous["label"])
    return matches.mean(), len(non_ambiguous)

rate, n = agreement_rate(clinical_unique)
print(f"Clinical: gold label agrees with CSV label {rate:.1%} of the time (on {n} non-ambiguous rows)")

rate, n = agreement_rate(diaries_unique)
print(f"Diaries: gold label agrees with CSV label {rate:.1%} of the time (on {n} non-ambiguous rows)")

Clinical: gold label agrees with CSV label 69.2% of the time (on 39 non-ambiguous rows)
Diaries: gold label agrees with CSV label 41.4% of the time (on 29 non-ambiguous rows)


In [16]:
# Clinical: what each mood word means clinically
clinical_word_map = {
    "low": "low mood", "fatigued": "fatigue", "sad": "sadness",
    "overwhelmed": "feeling overwhelmed", "unstable": "mood instability",
    "isolated": "social isolation", "anxious": "anxiety",
    "hopeful": "reports feeling hopeful", "better": "reports feeling better",
}
# Only these count as clinical "symptoms" (negative signal)
clinical_negative_words = {"low", "fatigued", "sad", "overwhelmed", "unstable", "isolated", "anxious"}

# Diary: what each second-clause phrase means
diary_clause_map = {
    "struggled to get out of bed and had no motivation": "low motivation, difficulty getting up",
    "felt overwhelmed with sadness and isolation": "sadness, feeling overwhelmed, isolation",
    "found it hard to focus today; it was a heavy day": "difficulty concentrating, low mood",
    "felt a bit off but managed to accomplish some tasks": "mild mood disturbance",
    "wrote in my journal and expressed my thoughts": None,  # neutral, no clinical content
    "felt a glimmer of hope today, looking forward to tomorrow": None,  # positive
    "took a walk in the evening, which slightly lifted my spirits": None,  # positive
    "shared a laugh with a friend, which helped me momentarily": None,  # positive
}

In [20]:
import re, json

def make_clinical_gold(row):
    text = row["text"]
    text_lower = text.lower()
    gold_label = row["gold_label"]
    symptoms = []
    follow_up = None

    if "demonstrated symptoms of depression" in text_lower or "shows signs of depression" in text_lower:
        symptoms.append("depressive symptoms")
        if "demonstrated symptoms" in text_lower:
            follow_up = "further assessment needed"

    # scan for any negative mood word, whole-word match, regardless of surrounding grammar
    for word in clinical_negative_words:
        if re.search(rf"\b{word}\b", text_lower):
            label = clinical_word_map[word]
            if label not in symptoms:
                symptoms.append(label)

    if "referral to a psychiatrist" in text_lower:
        follow_up = "referral to psychiatrist recommended"
    if "lack of interest in activities" in text_lower:
        symptoms.append("anhedonia (lack of interest in activities)")
    if "unable to concentrate" in text_lower:
        symptoms.append("difficulty concentrating")
    if "lacks motivation" in text_lower:
        symptoms.append("low motivation")
    if "needs support" in text_lower and follow_up is None:
        follow_up = "patient needs support"

    diagnosis = gold_label if gold_label in ("depression", "not depression") else None

    return {
        "chief_complaint": None,
        "symptoms": symptoms if symptoms else None,
        "diagnosis": diagnosis,
        "medical_history": None,
        "medications": None,
        "procedures": None,
        "follow_up": follow_up,
        "summary": text,
        "risk_indicators": "isolation" if "isolated" in text_lower else None,
        "urgency": "moderate" if follow_up else "low",
        "_gold_label_for_eval": gold_label
    }

clinical_unique["gold_json"] = clinical_unique.apply(make_clinical_gold, axis=1)

# re-check the exact 3 that broke
for idx in [clinical_unique[clinical_unique["text"].str.contains("experiencing low thoughts")].index[0],
            clinical_unique[clinical_unique["text"].str.contains("feelings of being anxious")].index[0],
            clinical_unique[clinical_unique["text"].str.contains("feeling better and fatigued")].index[0]]:
    print(clinical_unique.loc[idx, "text"])
    print(json.dumps(clinical_unique.loc[idx, "gold_json"], indent=2))
    print("=" * 60)

Patient has been experiencing low thoughts and needs support.
{
  "chief_complaint": null,
  "symptoms": [
    "low mood"
  ],
  "diagnosis": "depression",
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": "patient needs support",
  "summary": "Patient has been experiencing low thoughts and needs support.",
  "risk_indicators": null,
  "urgency": "moderate",
  "_gold_label_for_eval": "depression"
}
Patient expressed feelings of being anxious and lacks motivation.
{
  "chief_complaint": null,
  "symptoms": [
    "anxiety",
    "low motivation"
  ],
  "diagnosis": "depression",
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": null,
  "summary": "Patient expressed feelings of being anxious and lacks motivation.",
  "risk_indicators": null,
  "urgency": "low",
  "_gold_label_for_eval": "depression"
}
Patient shows signs of depression, feeling better and fatigued.
{
  "chief_complaint": null,
  "symptoms": [
    "dep

In [21]:
clinical_unique["gold_json"] = clinical_unique.apply(make_clinical_gold, axis=1)

sample = clinical_unique.sample(8, random_state=7)
for i, row in sample.iterrows():
    print(f"TEXT: {row['text']}")
    print(json.dumps(row['gold_json'], indent=2))
    print("=" * 60)

TEXT: Patient has been experiencing low thoughts and needs support.
{
  "chief_complaint": null,
  "symptoms": [
    "low mood"
  ],
  "diagnosis": "depression",
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": "patient needs support",
  "summary": "Patient has been experiencing low thoughts and needs support.",
  "risk_indicators": null,
  "urgency": "moderate",
  "_gold_label_for_eval": "depression"
}
TEXT: Patient expressed feelings of being anxious and lacks motivation.
{
  "chief_complaint": null,
  "symptoms": [
    "anxiety",
    "low motivation"
  ],
  "diagnosis": "depression",
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": null,
  "summary": "Patient expressed feelings of being anxious and lacks motivation.",
  "risk_indicators": null,
  "urgency": "low",
  "_gold_label_for_eval": "depression"
}
TEXT: Patient expressed feelings of being isolated and lacks motivation.
{
  "chief_complaint": null,
  "

In [22]:
clinical_unique.to_json("clinical_gold_eval_set.jsonl", orient="records", lines=True)
print("Saved", len(clinical_unique), "rows")

Saved 57 rows


In [23]:
diary_clause_signals = {
    "struggled to get out of bed and had no motivation": {"symptoms": ["low motivation", "difficulty getting out of bed"], "urgency": "moderate"},
    "felt overwhelmed with sadness and isolation": {"symptoms": ["sadness", "feeling overwhelmed", "social isolation"], "urgency": "moderate", "risk": "isolation"},
    "found it hard to focus today; it was a heavy day": {"symptoms": ["difficulty concentrating", "low mood"], "urgency": "low"},
    "felt a bit off but managed to accomplish some tasks": {"symptoms": ["mild mood disturbance"], "urgency": "low"},
    "wrote in my journal and expressed my thoughts": {"symptoms": None, "urgency": "low"},
    "felt a glimmer of hope today, looking forward to tomorrow": {"symptoms": None, "urgency": "low"},
    "took a walk in the evening, which slightly lifted my spirits": {"symptoms": None, "urgency": "low"},
    "shared a laugh with a friend, which helped me momentarily": {"symptoms": None, "urgency": "low"},
}

opener_negative_words = {"confused", "down", "empty", "overwhelmed"}

def make_diary_gold(row):
    text = row["text"]
    gold_label = row["gold_label"]

    opener_match = re.search(r"I felt (\w+) today", text)
    opener_word = opener_match.group(1) if opener_match else None

    clause_info = None
    for clause, info in diary_clause_signals.items():
        if clause in text:
            clause_info = info
            break

    symptoms = list(clause_info["symptoms"]) if clause_info and clause_info["symptoms"] else []
    if opener_word in opener_negative_words and opener_word not in [s.lower() for s in symptoms]:
        symptoms.append(f"reports feeling {opener_word}")

    diagnosis = gold_label if gold_label in ("depression", "not depression") else None

    return {
        "chief_complaint": None,
        "symptoms": symptoms if symptoms else None,
        "diagnosis": diagnosis,
        "medical_history": None,
        "medications": None,
        "procedures": None,
        "follow_up": None,
        "summary": text,
        "risk_indicators": clause_info.get("risk") if clause_info else None,
        "urgency": clause_info["urgency"] if clause_info else "low",
        "_gold_label_for_eval": gold_label
    }

diaries_unique["gold_json"] = diaries_unique.apply(make_diary_gold, axis=1)

sample = diaries_unique.sample(8, random_state=3)
for i, row in sample.iterrows():
    print(f"TEXT: {row['text']}")
    print(json.dumps(row['gold_json'], indent=2))
    print("=" * 60)

TEXT: I felt down today. I shared a laugh with a friend, which helped me momentarily.
{
  "chief_complaint": null,
  "symptoms": [
    "reports feeling down"
  ],
  "diagnosis": null,
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": null,
  "summary": "I felt down today. I shared a laugh with a friend, which helped me momentarily.",
  "risk_indicators": null,
  "urgency": "low",
  "_gold_label_for_eval": "ambiguous"
}
TEXT: I felt confused today. I wrote in my journal and expressed my thoughts.
{
  "chief_complaint": null,
  "symptoms": [
    "reports feeling confused"
  ],
  "diagnosis": null,
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": null,
  "summary": "I felt confused today. I wrote in my journal and expressed my thoughts.",
  "risk_indicators": null,
  "urgency": "low",
  "_gold_label_for_eval": "ambiguous"
}
TEXT: I felt hopeful today. I felt overwhelmed with sadness and isolation.
{
  "chief_compla

In [24]:
diaries_unique.to_json("diaries_gold_eval_set.jsonl", orient="records", lines=True)
print("Saved", len(diaries_unique), "rows")

Saved 64 rows


In [26]:
clinical_unique["source"] = "clinical"
diaries_unique["source"] = "diary"

eval_set = pd.concat([
    clinical_unique[["source", "text", "label", "gold_label", "gold_json"]],
    diaries_unique[["source", "text", "label", "gold_label", "gold_json"]]
], ignore_index=True)

eval_set.to_json("full_eval_set.jsonl", orient="records", lines=True)

print(f"Total eval set: {len(eval_set)} notes")
print(eval_set["gold_label"].value_counts())
print(eval_set.groupby("source")["gold_label"].value_counts())

Total eval set: 121 notes
gold_label
ambiguous         53
depression        50
not depression    18
Name: count, dtype: int64
source    gold_label    
clinical  depression        37
          ambiguous         18
          not depression     2
diary     ambiguous         35
          not depression    16
          depression        13
Name: count, dtype: int64


In [ ]:
#%pip install google-genai python-dotenv

  Using cached google_genai-2.23.0-py3-none-any.whl.metadata (56 kB)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached websockets-16.1.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.5-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
Using cached google_genai-2.23.0-py3-none-any.whl (1.1 MB)
Using cached pydantic-2.13.5-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.5-cp313-cp313-macosx_11_0_arm64.whl (1.9 MB)
Using cached websockets-16.1.1-cp313-cp313-macosx_11_0_arm64.whl (177 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 615.0 kB/s  0:00:07 eta 0:00:01
Using ca

In [90]:
import os, time
from dotenv import load_dotenv
from google import genai
from google.genai.errors import ClientError

load_dotenv()

api_keys = [
    os.environ.get("GEMINI_API_KEY_1"),
    os.environ.get("GEMINI_API_KEY_2"),
    os.environ.get("GEMINI_API_KEY_3"),
]
api_keys = [k for k in api_keys if k]  # drop any that aren't set
clients = [genai.Client(api_key=k) for k in api_keys]

print(f"Loaded {len(clients)} API key(s)")

_current_client_index = 0

def generate_with_fallback(**kwargs):
    global _current_client_index
    last_error = None
    
    for attempt in range(len(clients)):
        idx = (_current_client_index + attempt) % len(clients)
        try:
            response = clients[idx].models.generate_content(**kwargs)
            _current_client_index = idx  # stick with the working key next time
            return response
        except ClientError as e:
            if e.code == 429:
                print(f"Key #{idx+1} hit rate limit, trying next key...")
                last_error = e
                continue
            else:
                raise  # a non-rate-limit error should surface immediately, not be swallowed
    
    raise last_error  # all keys exhausted

Loaded 3 API key(s)


In [ ]:
V1 = """ I need your help with extracting some information from clinical notes and patient diaries. I will provide you with a text, and I want you to extract the following information,
and if there any filed that is not includes just put null, and also dont just guess or infer:
Return ONLY valid JSON matching this exact schema:
{
  "chief_complaint": string or null,
  "symptoms": array of strings or null,
  "diagnosis": string or null,
  "medical_history": string or null,
  "medications": array of strings or null,
  "procedures": array of strings or null,
  "follow_up": string or null,
  "summary": string,
  "risk_indicators": string or null,
  "urgency": "low" or "moderate" or "high"
}
 """

In [33]:
def extract_v1(note_text):
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=f"{V1}\n\nText:\n{note_text}"
    )
    return response.text

test_note = "Patient has fluctuating moods, feeling low."
print("INPUT:", test_note)
print("\nOUTPUT:")
print(extract_v1(test_note))

INPUT: Patient has fluctuating moods, feeling low.

OUTPUT:
It looks like you didn't specify which exact fields or categories you would like me to extract (e.g., Symptoms, Duration, Severity, Medications, etc.). 

Based on the brief text provided, here is a general structured extraction:

* **Primary Symptom / Subjective Complaint:** Fluctuating moods, feeling low
* **Clinical Domain:** Psychiatry / Mental Health
* **Affective State:** Low mood / Mood instability
* **Duration / Onset:** Not specified
* **Severity:** Not specified

***

**How would you like this formatted?** 
If you have a specific schema or list of entities you need extracted (e.g., JSON format, specific clinical entities like *Symptom*, *Severity*, *Timeline*), please let me know or provide the rest of the text, and I will tailor the output for you!


In [40]:
import json

# pick specific interesting cases + some random ones
clear_depression = eval_set[eval_set["gold_label"] == "depression"].iloc[0]
clear_not = eval_set[eval_set["gold_label"] == "not depression"].iloc[0]
ambiguous = eval_set[eval_set["gold_label"] == "ambiguous"].iloc[0]
random_two = eval_set.sample(2, random_state=11)

test_rows = pd.concat([
    pd.DataFrame([clear_depression]),
    pd.DataFrame([clear_not]),
    pd.DataFrame([ambiguous]),
    random_two
])

for i, row in test_rows.iterrows():
    print(f"GOLD LABEL: {row['gold_label']}")
    print(f"TEXT: {row['text']}")
    output = extract_v1(row['text'])
    print("MODEL OUTPUT:")
    print(output)
    print("=" * 70)

GOLD LABEL: depression
TEXT: Patient demonstrated symptoms of depression; further assessment needed.
MODEL OUTPUT:
It looks like you didn't list the specific categories or fields you would like extracted! 

However, based on the short text provided, here is a general extraction:

* **Symptom / Clinical Impression:** Symptoms of depression
* **Action / Plan:** Further assessment needed
* **Certainty / Status:** Suspected / Requires evaluation (not yet a confirmed diagnosis)

***

**How would you like me to structure the extraction going forward?** 
If you have a specific format or list of fields (for example: *Patient ID, Symptoms, Diagnosis, Plan, Medications, Dates*), please provide them, and I will extract the data accordingly!
GOLD LABEL: not depression
TEXT: Patient has fluctuating moods, feeling better.
MODEL OUTPUT:
Here is the structured information extracted from the provided text:

* **Symptom / Clinical Presentation:** Fluctuating moods
* **Current Status / Trend:** Feeling b

In [ ]:
V2 = """
    You are a data extractor. Extract only information that is explicitly stated in the text, and structure the output in this JSON format:
    {  "chief_complaint": null,  
        "symptoms": [],  
        "diagnosis": null,  
        "medical_history": [],  
        "medications": [    
            {   "name": null,      
                "dose": null,      
                "frequency": null,      
                "duration": null    }  
            ],  
        "procedures": [],  
        "follow_up": null,  
        "summary": null,  
        "risk_indicators": [],
        "urgency": null
    }
    If any field is not included just put null, and also don't just guess or infer.
    Output only the JSON object — no explanation, no commentary, no questions, no extra text.
"""

In [36]:
def extract_v2(note_text):
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=f"{V2}\n\nText:\n{note_text}"
    )
    return response.text

test_note = "Patient has fluctuating moods, feeling low."
print(extract_v2(test_note))

```json
{
  "chief_complaint": "fluctuating moods, feeling low",
  "symptoms": [
    "fluctuating moods",
    "feeling low"
  ],
  "diagnosis": [],
  "medical_history": [],
  "medications": [],
  "procedures": [],
  "follow_up": null,
  "summary": "Patient reports having fluctuating moods and feeling low. No diagnosis of depression is stated in the provided text.",
  "risk_indicators": [],
  "urgency": null
}
```


In [39]:
import json

# pick specific interesting cases + some random ones
clear_depression = eval_set[eval_set["gold_label"] == "depression"].iloc[0]
clear_not = eval_set[eval_set["gold_label"] == "not depression"].iloc[0]
ambiguous = eval_set[eval_set["gold_label"] == "ambiguous"].iloc[0]
random_two = eval_set.sample(2, random_state=11)

test_rows = pd.concat([
    pd.DataFrame([clear_depression]),
    pd.DataFrame([clear_not]),
    pd.DataFrame([ambiguous]),
    random_two
])

for i, row in test_rows.iterrows():
    print(f"GOLD LABEL: {row['gold_label']}")
    print(f"TEXT: {row['text']}")
    output = extract_v2(row['text'])
    print("MODEL OUTPUT:")
    print(output)
    print("=" * 70)

GOLD LABEL: depression
TEXT: Patient demonstrated symptoms of depression; further assessment needed.
MODEL OUTPUT:
```json
{
  "chief_complaint": null,
  "symptoms": [
    "depression"
  ],
  "diagnosis": [],
  "medical_history": [],
  "medications": [],
  "procedures": [],
  "follow_up": "further assessment needed",
  "summary": "Patient demonstrated symptoms of depression; further assessment needed.",
  "risk_indicators": [],
  "urgency": null
}
```
GOLD LABEL: not depression
TEXT: Patient has fluctuating moods, feeling better.
MODEL OUTPUT:
```json
{
  "chief_complaint": null,
  "symptoms": [
    "fluctuating moods"
  ],
  "diagnosis": [],
  "medical_history": [],
  "medications": [],
  "procedures": [],
  "follow_up": null,
  "summary": "Patient reports fluctuating moods but is currently feeling better. Based strictly on the provided text, there is insufficient information and no explicit medical diagnosis to determine if the patient has depression.",
  "risk_indicators": [],
  "ur

In [42]:
V3 = """
    Role : You are a Therapist.

    Task : Your task is to extract required information from the clinical notes i give you.

    Requirements :

    - Only use information that was mentioned.
    - Do not invent or infer missing information, even if it seems likely.
    - If a field is not mentioned, use null for that field. Do not use empty strings or empty arrays for missing data — use null consistently.
    - Use the exact field names and structure provided in the output format below.
    - Output ONLY the JSON object. Do not include any explanation, commentary, questions, or markdown formatting.

    Output Format :
    {  "chief_complaint": null,  
            "symptoms": [],  
            "diagnosis": null,  
            "medical_history": [],  
            "medications": [    
                {   "name": null,      
                    "dose": null,      
                    "frequency": null,      
                    "duration": null    }  
                ],  
            "procedures": [],  
            "follow_up": null,  
            "summary": null,  
            "risk_indicators": [],
            "urgency": null
    }

    as in JSON object
"""

In [45]:
def extract_v3(note_text):
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=f"{V3}\n\nText:\n{note_text}"
    )
    return response.text

for i, row in test_rows.iterrows():
    print(f"GOLD LABEL: {row['gold_label']}")
    print(f"TEXT: {row['text']}")
    output = extract_v3(row['text'])
    print("MODEL OUTPUT:")
    print(output)
    print("=" * 70)

GOLD LABEL: depression
TEXT: Patient demonstrated symptoms of depression; further assessment needed.
MODEL OUTPUT:
{  "chief_complaint": null,  
        "symptoms": ["depression"],  
        "diagnosis": null,  
        "medical_history": [],  
        "medications": [    
            {   "name": null,      
                "dose": null,      
                "frequency": null,      
                "duration": null    }  
            ],  
        "procedures": [],  
        "follow_up": "further assessment needed",  
        "summary": "Patient demonstrated symptoms of depression; further assessment needed.",  
        "risk_indicators": [],
        "urgency": null
}
GOLD LABEL: not depression
TEXT: Patient has fluctuating moods, feeling better.
MODEL OUTPUT:
{  "chief_complaint": null,  
        "symptoms": ["fluctuating moods"],  
        "diagnosis": null,  
        "medical_history": [],  
        "medications": [    
            {   "name": null,      
                "dose": nul

In [47]:
FINAL = """
    Role: You are a Therapist extracting structured data from clinical notes.

    Rules:
    - Only extract what is explicitly stated. Never infer or guess.
    - Always fill in "summary" with a one-sentence summary, even if nothing else is found.
    - Always set "urgency" to "low", "moderate", or "high" — never leave it blank.
    - Use null for any field with no information. Do not use empty arrays or placeholder objects.

    The output must be a valid JSON object with exactly these fields:
    {  "chief_complaint": null,  
            "symptoms": null,  
            "diagnosis": null,  
            "medical_history": null,  
            "medications": null,  
            "procedures": null,  
            "follow_up": null,  
            "summary": null,  
            "risk_indicators": null,
            "urgency": null
        }

    Example:
    Note: "Patient feels sad; a referral to a psychiatrist may be necessary."
    Output:
    {
    "chief_complaint": null,
    "symptoms": ["sadness"],
    "diagnosis": null,
    "medical_history": null,
    "medications": null,
    "procedures": null,
    "follow_up": "referral to psychiatrist recommended",
    "summary": "Patient reports sadness; psychiatric referral suggested.",
    "risk_indicators": null,
    "urgency": "moderate"
    }

    Now extract from this note, following the exact same format:
"""

In [ ]:
def extract_final(note_text):
    response = generate_with_fallback(
        model="gemini-3.5-flash-lite",
        contents=f"{FINAL}\n\nText:\n{note_text}"
    )
    return response.text

for i, row in test_rows.iterrows():
    print(f"GOLD LABEL: {row['gold_label']}")
    print(f"TEXT: {row['text']}")
    output = extract_final(row['text'])
    print("MODEL OUTPUT:")
    print(output)
    print("=" * 70)

GOLD LABEL: depression
TEXT: Patient demonstrated symptoms of depression; further assessment needed.
MODEL OUTPUT:
{
  "chief_complaint": null,
  "symptoms": [
    "depression"
  ],
  "diagnosis": null,
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": "further assessment needed",
  "summary": "Patient showed symptoms of depression and requires further assessment.",
  "risk_indicators": null,
  "urgency": "moderate"
}
GOLD LABEL: not depression
TEXT: Patient has fluctuating moods, feeling better.
MODEL OUTPUT:
{
  "chief_complaint": null,
  "symptoms": [
    "fluctuating moods"
  ],
  "diagnosis": null,
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": null,
  "summary": "Patient experiences fluctuating moods but reports feeling better overall.",
  "risk_indicators": null,
  "urgency": "low"
}
GOLD LABEL: ambiguous
TEXT: Patient indicates a lack of interest in activities, feeling hopeful.
MODEL OUTPUT:
```json
{


In [ ]:
import json, re, time

def parse_json_response(raw_text):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip())
    return json.loads(cleaned)

def run_full_batch(eval_set, extract_fn, delay=4):
    results = []
    for i, row in eval_set.iterrows():
        try:
            raw_output = extract_fn(row["text"])
            parsed = parse_json_response(raw_output)
            success = True
        except Exception as e:
            parsed = None
            success = False
            print(f"FAILED at row {i}: {e}")
        
        results.append({
            "index": i,
            "source": row["source"],
            "text": row["text"],
            "gold_label": row["gold_label"],
            "gold_json": row["gold_json"],
            "model_output": parsed,
            "parse_success": success
        })
        time.sleep(delay)  # stay under free-tier rate limits
        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(eval_set)}")
    return results

In [49]:
import json, re, time

def parse_json_response(raw_text):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip())
    return json.loads(cleaned)

def run_full_batch(eval_set, extract_fn, delay=4):
    results = []
    for i, row in eval_set.iterrows():
        try:
            raw_output = extract_fn(row["text"])
            parsed = parse_json_response(raw_output)
            success = True
        except Exception as e:
            parsed = None
            success = False
            print(f"FAILED at row {i}: {e}")
        
        results.append({
            "index": i,
            "source": row["source"],
            "text": row["text"],
            "gold_label": row["gold_label"],
            "gold_json": row["gold_json"],
            "model_output": parsed,
            "parse_success": success
        })
        time.sleep(delay)  # stay under free-tier rate limits
        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(eval_set)}")
    return results

In [50]:
batch_results = run_full_batch(eval_set, extract_final, delay=4)

parse_success_rate = sum(r["parse_success"] for r in batch_results) / len(batch_results)
print(f"\nJSON parse success rate: {parse_success_rate:.1%}")

Processed 10/121
Processed 20/121
Processed 30/121
Processed 40/121
Processed 50/121
Processed 60/121
Processed 70/121
Processed 80/121
Processed 90/121
Processed 100/121
Processed 110/121
Processed 120/121

JSON parse success rate: 100.0%


In [51]:
with open("final_prompt_results.jsonl", "w") as f:
    for r in batch_results:
        f.write(json.dumps(r) + "\n")
print("Saved", len(batch_results), "results")

Saved 121 results


In [53]:
import json

results = []
with open("final_prompt_results.jsonl") as f:
    for line in f:
        results.append(json.loads(line))

print(f"Loaded {len(results)} results")

Loaded 121 results


In [54]:
literal_depression_rows = [r for r in results if 'depression' in r['text'].lower()]
print(f"Notes literally containing the word 'depression': {len(literal_depression_rows)}\n")

for r in literal_depression_rows:
    model_diag = r['model_output'].get('diagnosis') if r['model_output'] else None
    print(f"TEXT: {r['text']}")
    print(f"MODEL diagnosis field: {model_diag}")
    print("-" * 50)

Notes literally containing the word 'depression': 9

TEXT: Patient demonstrated symptoms of depression; further assessment needed.
MODEL diagnosis field: None
--------------------------------------------------
TEXT: Patient shows signs of depression, feeling low and fatigued.
MODEL diagnosis field: None
--------------------------------------------------
TEXT: Patient shows signs of depression, feeling isolated and fatigued.
MODEL diagnosis field: None
--------------------------------------------------
TEXT: Patient shows signs of depression, feeling anxious and fatigued.
MODEL diagnosis field: None
--------------------------------------------------
TEXT: Patient shows signs of depression, feeling better and fatigued.
MODEL diagnosis field: None
--------------------------------------------------
TEXT: Patient shows signs of depression, feeling sad and fatigued.
MODEL diagnosis field: None
--------------------------------------------------
TEXT: Patient shows signs of depression, feeling

In [ ]:
#%pip install requests


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [57]:
import requests
import xml.etree.ElementTree as ET

def condition_info_lookup(condition_text, max_results=1):
    url = "https://wsearch.nlm.nih.gov/ws/query"
    params = {"db": "healthTopics", "term": condition_text, "rettype": "brief"}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    
    root = ET.fromstring(response.content)
    results = []
    for doc in root.findall(".//document")[:max_results]:
        title = doc.find(".//content[@name='title']")
        summary = doc.find(".//content[@name='snippet']")
        results.append({
            "title": title.text if title is not None else None,
            "summary": summary.text if summary is not None else None,
            "url": doc.attrib.get("url")
        })
    return results

In [58]:
result = condition_info_lookup("depression")
print(result)

[{'title': '<span class="qt0">Depression</span>', 'summary': 'What is <span class="qt0">depression? Depression</span> is more than a feeling of being sad or irritable for a few days. It\'s a serious mood disorder. As one of the most common  ... ', 'url': 'https://medlineplus.gov/depression.html'}]


In [59]:
import re

def clean_html(text):
    if text is None:
        return None
    return re.sub(r"<[^>]+>", "", text).strip()

def condition_info_lookup(condition_text, max_results=1):
    url = "https://wsearch.nlm.nih.gov/ws/query"
    params = {"db": "healthTopics", "term": condition_text, "rettype": "brief"}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    
    root = ET.fromstring(response.content)
    results = []
    for doc in root.findall(".//document")[:max_results]:
        title = doc.find(".//content[@name='title']")
        summary = doc.find(".//content[@name='snippet']")
        results.append({
            "title": clean_html(title.text if title is not None else None),
            "summary": clean_html(summary.text if summary is not None else None),
            "url": doc.attrib.get("url")
        })
    return results

# re-test
print(condition_info_lookup("depression"))

[{'title': 'Depression', 'summary': "What is depression? Depression is more than a feeling of being sad or irritable for a few days. It's a serious mood disorder. As one of the most common  ...", 'url': 'https://medlineplus.gov/depression.html'}]


In [60]:
print(condition_info_lookup("anxiety"))

[{'title': 'Anxiety', 'summary': 'What is anxiety? Anxiety is a feeling of fear, dread, and uneasiness. It might cause you to sweat, feel restless and tense, and have a rapid heartbeat.    ...', 'url': 'https://medlineplus.gov/anxiety.html'}]


In [ ]:
#%pip install mcp

  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached starlette-1.6.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached uvicorn-0.52.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached idna-3.19-py3-none-any.whl.metadata (9.2 kB)
Using cached idna-3.19-py3-none-any.whl (68 kB)
Using cached python_multipart-0.0.32-py3-none-any.whl (30 kB)
Using cached starlette-1.6.0-py3-none-any.whl (75 kB)
Using cached uvicorn-0.52.4-py3-none-any.whl (79 kB)
  Attempting uninstall: idna0m╺━━━━━━━━━━━━━━━━━━━━━━━━━━  4/12 [opentelemetry-api]
    Found existing installation: idna 3.10━━━━━━━━━━━━━━━━━━━━  4/12 [opentelemetry-api]
    Uninstalling idna-3.10:━━━━━━━━━━━━━━━━━━━━━━━━━━  4/12 [opentelemetry-api]
      Successfully uninstalled idna-3.10━━━━━━━━━━━━━━━━━━━━━━  4/12 [opentelemetry-api]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [mcp]32m11/12 [mcp]x2]te]api]

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip insta

In [63]:
from mcp.server import MCPServer

mcp_server = MCPServer("medextract-tools")

@mcp_server.tool()
def condition_info_lookup_tool(condition_text: str) -> dict:
    """Look up authoritative medical information about a condition from MedlinePlus.
    Use this to verify or ground a diagnosis before stating it — never rely on memory alone.
    
    Args:
        condition_text: the name of the medical condition to look up (e.g. 'depression')
    
    Returns:
        A dict with the condition's title, a summary description, and a source URL.
    """
    results = condition_info_lookup(condition_text, max_results=1)
    if not results:
        return {"found": False, "message": f"No MedlinePlus entry found for '{condition_text}'"}
    return {"found": True, **results[0]}

In [64]:
print(condition_info_lookup_tool("depression"))

{'found': True, 'title': 'Depression', 'summary': "What is depression? Depression is more than a feeling of being sad or irritable for a few days. It's a serious mood disorder. As one of the most common  ...", 'url': 'https://medlineplus.gov/depression.html'}


In [65]:
def medication_lookup(drug_name):
    url = "https://rxnav.nlm.nih.gov/REST/rxcui.json"
    params = {"name": drug_name}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    data = response.json()
    
    rxcui_list = data.get("idGroup", {}).get("rxnormId")
    if not rxcui_list:
        return {"found": False, "message": f"'{drug_name}' is not a recognized medication in RxNorm"}
    return {"found": True, "rxcui": rxcui_list[0], "name": drug_name}


def dosing_lookup(drug_name):
    url = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
    params = {"drug_name": drug_name}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    data = response.json()
    
    if not data.get("data"):
        return {"found": False, "message": f"No DailyMed label found for '{drug_name}'"}
    return {
        "found": True,
        "note": "Standard reference dose from FDA label — not patient-specific, for clinician review.",
        "setid": data["data"][0].get("setid"),
        "title": data["data"][0].get("title")
    }

In [66]:
print("Medication lookup (real drug):", medication_lookup("sertraline"))
print("\nMedication lookup (fake drug — should fail):", medication_lookup("zorblatamine"))
print("\nDosing lookup:", dosing_lookup("sertraline"))

Medication lookup (real drug): {'found': True, 'rxcui': '36437', 'name': 'sertraline'}

Medication lookup (fake drug — should fail): {'found': False, 'message': "'zorblatamine' is not a recognized medication in RxNorm"}

Dosing lookup: {'found': True, 'note': 'Standard reference dose from FDA label — not patient-specific, for clinician review.', 'setid': '7d598f1a-6c76-458a-a155-7a6a0743a2d3', 'title': 'SERTRALINE HYDROCHLORIDE TABLET [REMEDYREPACK INC.]'}


In [67]:
@mcp_server.tool()
def medication_lookup_tool(drug_name: str) -> dict:
    """Verify a medication name is real before suggesting it. Never suggest a medication
    without calling this first — if it returns found: False, do not include that medication
    in the output.
    
    Args:
        drug_name: the medication name to verify (e.g. 'sertraline')
    """
    return medication_lookup(drug_name)


@mcp_server.tool()
def dosing_lookup_tool(drug_name: str) -> dict:
    """Get the standard FDA label dosing reference for a medication that has already
    been verified as real via medication_lookup_tool. This is a generic reference dose,
    never a personalized calculation — always present it to the clinician as such.
    
    Args:
        drug_name: the verified medication name to look up dosing for
    """
    return dosing_lookup(drug_name)

In [68]:
def icd10_lookup(diagnosis_text, max_results=3):
    url = "https://clinicaltables.nlm.nih.gov/api/icd10cm/v3/search"
    params = {"sf": "code,name", "terms": diagnosis_text, "maxList": max_results}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    data = response.json()
    
    results = [{"code": code, "description": name} for code, name in data[3]]
    if not results:
        return {"found": False, "message": f"No ICD-10 code found for '{diagnosis_text}'"}
    return {"found": True, "codes": results}

In [69]:
print(icd10_lookup("depression"))

{'found': True, 'codes': [{'code': 'F32.A', 'description': 'Depression, unspecified'}, {'code': 'F53.0', 'description': 'Postpartum depression'}, {'code': 'P91.4', 'description': 'Neonatal cerebral depression'}]}


In [70]:
@mcp_server.tool()
def icd10_lookup_tool(diagnosis_text: str) -> dict:
    """Look up the official ICD-10-CM code for a diagnosis that has already been confirmed
    via condition_info_lookup_tool. Never state an ICD-10 code from memory — always call
    this tool to get the real code.
    
    Args:
        diagnosis_text: the confirmed diagnosis to find a code for (e.g. 'depression')
    """
    return icd10_lookup(diagnosis_text)

In [73]:
def condition_info_lookup(condition_text: str, max_results: int = 1) -> dict:
    url = "https://wsearch.nlm.nih.gov/ws/query"
    params = {"db": "healthTopics", "term": condition_text, "rettype": "brief"}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    root = ET.fromstring(response.content)
    results = []
    for doc in root.findall(".//document")[:max_results]:
        title = doc.find(".//content[@name='title']")
        summary = doc.find(".//content[@name='snippet']")
        results.append({
            "title": clean_html(title.text if title is not None else None),
            "summary": clean_html(summary.text if summary is not None else None),
            "url": doc.attrib.get("url")
        })
    return results


def medication_lookup(drug_name: str) -> dict:
    url = "https://rxnav.nlm.nih.gov/REST/rxcui.json"
    params = {"name": drug_name}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    data = response.json()
    rxcui_list = data.get("idGroup", {}).get("rxnormId")
    if not rxcui_list:
        return {"found": False, "message": f"'{drug_name}' is not a recognized medication in RxNorm"}
    return {"found": True, "rxcui": rxcui_list[0], "name": drug_name}


def dosing_lookup(drug_name: str) -> dict:
    url = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
    params = {"drug_name": drug_name}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    data = response.json()
    if not data.get("data"):
        return {"found": False, "message": f"No DailyMed label found for '{drug_name}'"}
    return {
        "found": True,
        "note": "Standard reference dose from FDA label — not patient-specific, for clinician review.",
        "setid": data["data"][0].get("setid"),
        "title": data["data"][0].get("title")
    }


def icd10_lookup(diagnosis_text: str, max_results: int = 3) -> dict:
    url = "https://clinicaltables.nlm.nih.gov/api/icd10cm/v3/search"
    params = {"sf": "code,name", "terms": diagnosis_text, "maxList": max_results}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    data = response.json()
    results = [{"code": code, "description": name} for code, name in data[3]]
    if not results:
        return {"found": False, "message": f"No ICD-10 code found for '{diagnosis_text}'"}
    return {"found": True, "codes": results}

In [74]:
condition_info_lookup.__doc__ = "Look up authoritative medical info about a condition from MedlinePlus. Use this to verify a diagnosis before stating it."
medication_lookup.__doc__ = "Verify a medication name is real before suggesting it. If found is False, do not suggest that medication."
dosing_lookup.__doc__ = "Get standard FDA label dosing for an already-verified medication. This is a generic reference, never a personalized dose."
icd10_lookup.__doc__ = "Look up the official ICD-10-CM code for a confirmed diagnosis. Never state a code from memory."

In [ ]:
DIAGNOSIS_PROMPT = """You are reasoning about a possible diagnosis based on extracted symptoms.

Rules:
- Before stating any diagnosis, call condition_info_lookup_tool to verify it against real medical information.
- Before suggesting any medication, call medication_lookup_tool to confirm it's a real medication.
- If you suggest a medication, call dosing_lookup_tool to get the standard reference dose.
- Once a diagnosis is confirmed, call icd10_lookup_tool to get its official code.
- Never state a diagnosis, medication, dose, or code without calling the matching tool first.
- Clearly label medication/dosing suggestions as "for clinician review" — not a final prescription.
"""

def reason_diagnosis(symptoms_text):
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=f"{DIAGNOSIS_PROMPT}\n\nExtracted symptoms: {symptoms_text}",
        config={
            "tools": [
                condition_info_lookup,
                medication_lookup,
                dosing_lookup,
                icd10_lookup,
            ]
        }
    )
    return response.text

In [76]:
result = reason_diagnosis("feeling low, fatigue, loss of interest in activities")
print(result)

[09/12/26 22:21:06] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=717925;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=556982;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/12/26 22:21:07] INFO     HTTP Request: POST                                                     ]8;id=325311;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=736890;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/12/26 22:21:09] INFO     AFC remote call 1 is done.                                              ]8;id=177278;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=897307;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=985242;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=252297;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/12/26 22:21:11] INFO     AFC remote call 2 is done.                                              ]8;id=495397;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=708987;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/12/26 22:21:12] INFO     HTTP Request: POST                                                     ]8;id=820043;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=291869;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/12/26 22:21:19] INFO     AFC remote call 3 is done.                                              ]8;id=979678;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=905357;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/12/26 22:21:21] INFO     HTTP Request: POST                                                     ]8;id=420591;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=328784;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

Based on the symptoms you provided—feeling low, persistent fatigue, and a loss of interest in activities (anhedonia)—these are key clinical indicators consistent with **Major Depressive Disorder**. 

According to official medical classification (ICD-10-CM), the corresponding diagnosis code is **F32.1** (Major depressive disorder, single episode, moderate).

### Next Steps and Recommendations
* **Professional Evaluation:** A formal diagnosis should always be confirmed by a qualified healthcare provider or mental health professional through a comprehensive psychological evaluation.
* **Treatment Options:** Management often involves a combination of psychotherapy (such as cognitive behavioral therapy) and, when appropriate, pharmacotherapy. 

*(Note: Medication and dosing choices must be evaluated by a licensed clinician based on a complete medical history. No specific medication or dosage is prescribed here.)*


In [77]:
result = reason_diagnosis("feeling low, fatigue, loss of interest in activities")

# check the full response object, not just .text, for actual function call history
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=f"{DIAGNOSIS_PROMPT}\n\nExtracted symptoms: feeling low, fatigue, loss of interest in activities",
    config={"tools": [condition_info_lookup, medication_lookup, dosing_lookup, icd10_lookup]}
)

print("=== Automatic function calling history ===")
if hasattr(response, "automatic_function_calling_history") and response.automatic_function_calling_history:
    for entry in response.automatic_function_calling_history:
        print(entry)
else:
    print("No function calling history found on the response object.")

print("\n=== Candidates / function call parts ===")
for candidate in response.candidates:
    for part in candidate.content.parts:
        if hasattr(part, "function_call") and part.function_call:
            print("CALLED:", part.function_call.name, part.function_call.args)
        if hasattr(part, "function_response") and part.function_response:
            print("RESULT:", part.function_response.response)

[09/12/26 22:22:06] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=497947;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=720969;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=578228;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=776084;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/12/26 22:22:08] INFO     AFC remote call 1 is done.                                              ]8;id=820501;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=780398;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=162483;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=483103;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/12/26 22:22:09] INFO     AFC remote call 2 is done.                                              ]8;id=679010;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=80857;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

...

[09/12/26 22:22:12] INFO     AFC remote call 1 is done.                                              ]8;id=626119;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=908215;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/12/26 22:22:13] INFO     HTTP Request: POST                                                     ]8;id=442625;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=171187;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

                    INFO     AFC remote call 2 is done.                                              ]8;id=130685;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=956416;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/12/26 22:22:14] INFO     HTTP Request: POST                                                     ]8;id=806062;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=934655;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

=== Automatic function calling history ===
parts=[Part(
  text="""You are reasoning about a possible diagnosis based on extracted symptoms.

Rules:
- Before stating any diagnosis, call condition_info_lookup_tool to verify it against real medical information.
- Before suggesting any medication, call medication_lookup_tool to confirm it's a real medication.
- If you suggest a medication, call dosing_lookup_tool to get the standard reference dose.
- Once a diagnosis is confirmed, call icd10_lookup_tool to get its official code.
- Never state a diagnosis, medication, dose, or code without calling the matching tool first.
- Clearly label medication/dosing suggestions as "for clinician review" — not a final prescription.


Extracted symptoms: feeling low, fatigue, loss of interest in activities"""
)] role='user'
parts=[Part(
  function_call=FunctionCall(
    args={
      'condition_text': 'Major Depressive Disorder'
    },
    id='call_2302742',
    name='condition_info_lookup'
  ),
  though

In [78]:
result2 = reason_diagnosis("feeling low, fatigue, and I've been taking my usual zorbatrex for it")
print(result2)

[09/12/26 22:23:11] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=259531;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=68685;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/12/26 22:23:12] INFO     HTTP Request: POST                                                     ]8;id=496179;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=380588;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/12/26 22:23:14] INFO     AFC remote call 1 is done.                                              ]8;id=641380;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=406826;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/12/26 22:23:15] INFO     HTTP Request: POST                                                     ]8;id=637882;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=657214;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

It appears that "zorbatrex" is not recognized as a standard medication in medical databases. Because you are experiencing feelings of low mood and fatigue, it is very important to consult with a healthcare professional for a proper evaluation and guidance. 

To help provide more relevant information, could you share a bit more about how long you've been feeling this way? Sharing any other symptoms or what led you to take "zorbatrex" would also be helpful.


In [ ]:
from pydantic import BaseModel
from typing import Optional, List

class DiagnosisReasoningResult(BaseModel):
    diagnosis: Optional[str] = None
    icd10_code: Optional[str] = None
    icd10_description: Optional[str] = None
    suggested_medications: Optional[List[str]] = None
    unverified_medications_mentioned: Optional[List[str]] = None
    dosing_reference: Optional[str] = None
    confidence: str  # "low", "moderate", "high"
    reasoning_notes: str

DIAGNOSIS_PROMPT = """You are reasoning about a possible diagnosis based on extracted symptoms or note content.

Rules:
- Before stating any diagnosis, call condition_info_lookup_tool to verify it against real medical information.
- Before suggesting any medication, call medication_lookup_tool to confirm it's a real medication.
  - If a mentioned medication is NOT found/verified, do NOT ask the user clarifying questions.
    Instead, put its name in "unverified_medications_mentioned" and do not include it in "suggested_medications".
- If you suggest a verified medication, call dosing_lookup_tool and put the standard reference dose
  (clearly noting it is not patient-specific) in "dosing_reference".
- Once a diagnosis is confirmed via the tool, call icd10_lookup_tool to get its official code.
- Never state a diagnosis, medication, dose, or code without calling the matching tool first.
- If evidence is weak or mixed, set "confidence" to "low" and explain why in "reasoning_notes" —
  do not omit a response just because you're uncertain.
- Always respond with the structured output — never ask the user a clarifying question instead.
"""

def reason_diagnosis_v2(input_text):
    response = generate_with_fallback(
        model="gemini-3.5-flash-lite",
        contents=f"{DIAGNOSIS_PROMPT}\n\nInput: {input_text}",
        config={
            "tools": [condition_info_lookup, medication_lookup, dosing_lookup, icd10_lookup],
            "response_mime_type": "application/json",
            "response_schema": DiagnosisReasoningResult,
        }
    )
    return response.text

In [81]:
print("=== Test 1: clear case ===")
print(reason_diagnosis_v2("feeling low, fatigue, loss of interest in activities"))

print("\n=== Test 2: fake medication ===")
print(reason_diagnosis_v2("feeling low, fatigue, and I've been taking my usual zorbatrex for it"))

=== Test 1: clear case ===


[09/12/26 22:26:04] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=345047;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=928758;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/12/26 22:26:06] INFO     HTTP Request: POST                                                     ]8;id=534674;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=190900;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

                    WARNING  Warning: there are non-text parts in the response: ['function_call'],    ]8;id=116825;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/types.py\types.py]8;;\:]8;id=760918;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/types.py#8664\8664]8;;\
                             returning concatenated text result from text parts. Check the full                    
                             candidates.content.parts accessor to get the full model response.                     

                    INFO     AFC remote call 1 is done.                                              ]8;id=728748;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=611534;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

...

[09/12/26 22:26:23] INFO     AFC remote call 4 is done.                                              ]8;id=870596;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=31179;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=939239;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=341092;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/12/26 22:26:24] INFO     AFC remote call 5 is done.                                              ]8;id=851803;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=654524;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/12/26 22:26:25] INFO     HTTP Request: POST                                                     ]8;id=89178;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=678388;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

{
  "diagnosis": "Major Depressive Disorder",
  "icd10_code": "F33.0",
  "icd10_description": "Major depressive disorder, recurrent, mild",
  "suggested_medications": [
    "sertraline"
  ],
  "unverified_medications_mentioned": [
    "zorbatrex"
  ],
  "dosing_reference": "Standard reference dose from FDA label — not patient-specific, for clinician review.",
  "confidence": "low",
  "reasoning_notes": "The patient reports feeling low and fatigue, which are symptoms of depression, but the presentation is limited and the only medication mentioned (zorbatrex) was unverified."
}


In [82]:
sample_depression = eval_set[eval_set["gold_label"] == "depression"].sample(6, random_state=5)
sample_not = eval_set[eval_set["gold_label"] == "not depression"].sample(6, random_state=5)
sample_ambiguous = eval_set[eval_set["gold_label"] == "ambiguous"].sample(6, random_state=5)

phase35_sample = pd.concat([sample_depression, sample_not, sample_ambiguous]).reset_index(drop=True)
print(f"Sample size: {len(phase35_sample)}")

Sample size: 18


In [91]:
import time, json

phase35_results = []

for i, row in phase35_sample.iterrows():
    try:
        raw = reason_diagnosis_v2(row["text"])
        parsed = json.loads(raw)
        success = True
    except Exception as e:
        parsed = None
        success = False
        print(f"FAILED at row {i}: {e}")
    
    phase35_results.append({
        "text": row["text"],
        "gold_label": row["gold_label"],
        "model_output": parsed,
        "parse_success": success
    })
    
    if parsed:
        meds = parsed.get("suggested_medications")
        dosing = parsed.get("dosing_reference")
        dosing_flag = ""
        if meds and not dosing:
            dosing_flag = "  <-- BUG: medication suggested but no dosing_reference!"
        print(f"Done {i+1}/{len(phase35_sample)}: gold={row['gold_label']}, diagnosis={parsed.get('diagnosis')}, meds={meds}, dosing={dosing}{dosing_flag}")
    else:
        print(f"Done {i+1}/{len(phase35_sample)}: ERROR")
    
    time.sleep(5)

[09/13/26 10:30:59] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=201919;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=283360;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/13/26 10:31:00] INFO     HTTP Request: POST                                                     ]8;id=174299;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=690095;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/13/26 10:31:02] INFO     AFC remote call 1 is done.                                              ]8;id=823724;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=752545;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=594507;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=977404;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/13/26 10:31:04] INFO     AFC remote call 2 is done.                                              ]8;id=574983;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=306320;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

...

[09/13/26 10:34:23] INFO     AFC remote call 3 is done.                                              ]8;id=192154;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=325762;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 10:34:36] INFO     HTTP Request: POST                                                     ]8;id=653443;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=718007;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/13/26 10:34:37] INFO     AFC remote call 4 is done.                                              ]8;id=28460;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=581377;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 10:34:47] INFO     HTTP Request: POST                                                     ]8;id=107993;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=425686;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

Done 18/18: gold=ambiguous, diagnosis=Major Depressive Disorder, meds=['sertraline'], dosing=Standard starting dose for adults with major depressive disorder is 50 mg once daily (Note: not patient-specific, for clinician review).


In [92]:
with open("phase35_batch_results.jsonl", "w") as f:
    for r in phase35_results:
        f.write(json.dumps(r) + "\n")
print("Saved", len(phase35_results))

Saved 18


In [88]:
def dosing_lookup(drug_name: str) -> dict:
    url = "https://api.fda.gov/drug/label.json"
    params = {"search": f'openfda.generic_name:"{drug_name}"', "limit": 1}
    response = requests.get(url, params=params, timeout=5)
    if response.status_code == 404:
        return {"found": False, "message": f"No FDA label found for '{drug_name}'"}
    response.raise_for_status()
    data = response.json()
    
    if not data.get("results"):
        return {"found": False, "message": f"No FDA label found for '{drug_name}'"}
    
    label = data["results"][0]
    dosage_text = label.get("dosage_and_administration", [None])[0]
    
    if not dosage_text:
        return {"found": True, "dosage_text": None, "note": "Label found but no dosage section available."}
    
    return {
        "found": True,
        "dosage_text": dosage_text[:500],  # real text from the FDA label
        "note": "Standard reference dose from FDA label — not patient-specific, for clinician review."
    }

In [89]:
print(dosing_lookup("sertraline"))

{'found': True, 'dosage_text': 'DOSAGE AND ADMINISTRATION Initial Treatment Dosage for Adults Major Depressive Disorder –Sertraline hydrochloride treatment should be administered at a dose of 50 mg once daily. While a relationship between dose and effect has not been established for major depressive disorder, OCD, panic disorder, PTSD or social anxiety disorder, patients were dosed in a range of 50-200 mg/day in the clinical trials demonstrating the effectiveness of Sertraline hydrochloride for the treatment of this indication', 'note': 'Standard reference dose from FDA label — not patient-specific, for clinician review.'}


In [93]:
import json

def validate_and_repair(raw_json_text, schema_class, generate_fn, max_retries=2):
    for attempt in range(max_retries + 1):
        try:
            data = json.loads(raw_json_text)
            validated = schema_class(**data)
            return validated.model_dump(), True
        except Exception as e:
            if attempt == max_retries:
                return {"error": str(e), "raw_output": raw_json_text}, False
            raw_json_text = generate_fn(
                f"Your previous output failed validation with this error: {e}\n"
                f"Previous output: {raw_json_text}\n"
                f"Please fix it and return only valid JSON matching the required schema."
            ).text

In [96]:
from pydantic import BaseModel
from typing import Optional, List, Literal

class ExtractionSchema(BaseModel):
    chief_complaint: Optional[str] = None
    symptoms: Optional[List[str]] = None
    diagnosis: Optional[str] = None
    medical_history: Optional[str] = None
    medications: Optional[List[str]] = None
    procedures: Optional[List[str]] = None
    follow_up: Optional[str] = None
    summary: str
    risk_indicators: Optional[str] = None
    urgency: Literal["low", "moderate", "high"]

In [97]:
def generate_fn_for_repair(followup_prompt):
    return generate_with_fallback(model="gemini-3.5-flash-lite", contents=followup_prompt)

def extract_final_validated(note_text):
    raw_output = extract_final(note_text)  # your existing function, unchanged
    validated_data, success = validate_and_repair(
        raw_json_text=raw_output,
        schema_class=ExtractionSchema,   # the Pydantic model from Phase 4 setup
        generate_fn=generate_fn_for_repair,
        max_retries=2
    )
    return validated_data, success

In [98]:
# normal case — should just pass validation cleanly
data, ok = extract_final_validated("Patient has fluctuating moods, feeling sad.")
print("Success:", ok)
print(data)

print("\n" + "="*50 + "\n")

# now let's PROVE the repair loop actually works by feeding it broken JSON directly
broken_json = '{"chief_complaint": null, "symptoms": ["sad"], "summary": "test"}'  # missing required "urgency"

def fake_generate_fn(prompt):
    print("--- REPAIR LOOP TRIGGERED, asking model to fix ---")
    return generate_with_fallback(model="gemini-3.5-flash-lite", contents=prompt)

fixed_data, ok = validate_and_repair(broken_json, ExtractionSchema, fake_generate_fn, max_retries=2)
print("Success after repair:", ok)
print(fixed_data)

[09/13/26 12:05:47] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=634240;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=649783;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/13/26 12:05:48] INFO     HTTP Request: POST                                                     ]8;id=175420;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=90024;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=558216;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=985338;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/13/26 12:05:49] INFO     HTTP Request: POST                                                     ]8;id=414790;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=942464;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

Success: True
{'chief_complaint': None, 'symptoms': ['fluctuating moods', 'sadness'], 'diagnosis': None, 'medical_history': None, 'medications': None, 'procedures': None, 'follow_up': None, 'summary': 'Patient presents with fluctuating moods and feelings of sadness.', 'risk_indicators': None, 'urgency': 'low'}


--- REPAIR LOOP TRIGGERED, asking model to fix ---


...

[09/13/26 12:05:50] INFO     HTTP Request: POST                                                     ]8;id=720720;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=815175;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

--- REPAIR LOOP TRIGGERED, asking model to fix ---


                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=449843;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=629956;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/13/26 12:05:51] INFO     HTTP Request: POST                                                     ]8;id=670782;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=49449;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

Success after repair: True
{'chief_complaint': 'sad', 'symptoms': ['sad'], 'diagnosis': None, 'medical_history': None, 'medications': None, 'procedures': None, 'follow_up': None, 'summary': 'test', 'risk_indicators': None, 'urgency': 'low'}
